In [24]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")   # geocoding_ml/ -> lets us import from helpers/

import pandas as pd

from helpers.pairs import load_pairs, add_number_street
from helpers.features import get_processed_dfs
from helpers.check_model_performance import review_typical_slices, find_random_sample_given_mask
from model_baseline.process_dumb_models import run_against_models

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
DATASET = "overture_case2s"   # or "overture" (the old set)
# Feature-cache prefix per dataset: each dataset has its own cache files
CACHE = {"overture": "./overture_training/overture_evaluation",
         "overture_case2s": "./case2s_training/case2s_evaluation"}[DATASET]

ot_train, ot_train_label = load_pairs(DATASET, "train")
ot_val, ot_val_label = load_pairs(DATASET, "val")
ot_test, ot_test_label = load_pairs(DATASET, "test")

# House number + street remainder for the similarity features (helpers/pairs.py):
# adds number_a/b, street_a/b, number_missing, both_number_missing
ot_train, ot_val, ot_test = [add_number_street(df) for df in [ot_train, ot_val, ot_test]]

In [26]:
# Full feature build (cached; ~1.5 min on first run)
train_overture, val_overture, test_overture = get_processed_dfs([
    [ot_train, f"{CACHE}_train.parquet"],
    [ot_val, f"{CACHE}_val.parquet"],
    [ot_test, f"{CACHE}_test.parquet"],
])

In [27]:
train_overture.columns

Index(['jaccard_street', 'jaccard_zip', 'jaccard_address', 'sorenson_street',
       'sorenson_zip', 'sorenson_address', 'overlap_street', 'overlap_zip',
       'overlap_address', 'tversky_street', 'tversky_zip', 'tversky_address',
       'cosine_street', 'cosine_zip', 'cosine_address', 'jw_street_sim',
       'jw_zipcode_sim', 'jw_address_sim', 'lv_street_sim', 'lv_zipcode_sim',
       'lv_address_sim', 'hm_zipcode_sim', 'number_missing'],
      dtype='object')

In [28]:
# Overture -> Overture, scored on val
ret, models = run_against_models(train_overture, ot_train_label, val_overture, test_overture, return_models=True)

In [29]:
ret.keys()

dict_keys(['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier'])

In [30]:
for model in ['LogisticRegression', 'RandomForestClassifier', 'XGBClassifier']:
    print(f"{'='*20} {model} {'='*20}")
    print(review_typical_slices(ot_val, ret[model]["val"][0], ot_val_label))
    print()


==================== LogisticRegression ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      455     342     57.1%
2     neg: case1 (same street, diff number)  13781    13673     108     99.2%
3  neg: case2 (same number, similar street)     34       16      18     47.1%
4                         neg: case3 (easy)  22969    22965       4    100.0%
5   neg: case2s (swapped look-alike street)    894      688     206     77.0%
6                        overall (accuracy)  41865    41187     678     98.4%

==================== RandomForestClassifier ====================
                                      slice      n  correct  errors % correct
0                      positives: identical   3390     3390       0    100.0%
1                  positives: non-identical    797      612     185     76.8%
2     neg: case

### Ablation: missing house number read as "0"

Flag `missing_number_as_zero=True` in `get_processed_dfs`: a missing number becomes `"0"` and the 7 number features are computed normally — no −1 / 1 overrides, no `number_missing` column. Same three models; results in `ret_zero`, compared to `ret` (val) below. Features cached separately (`*_missing_as_zero.parquet`).

In [31]:
# Flag: True = missing house number read as "0", no explicit number-missing signal
MISSING_NUMBER_AS_ZERO = True
train_zero, val_zero, test_zero = get_processed_dfs([
    [ot_train, f"{CACHE}_train_missing_as_zero.parquet"],
    [ot_val, f"{CACHE}_val_missing_as_zero.parquet"],
    [ot_test, f"{CACHE}_test_missing_as_zero.parquet"],
], missing_number_as_zero=MISSING_NUMBER_AS_ZERO)
ret_zero, models_zero = run_against_models(train_zero, ot_train_label, val_zero, test_zero, return_models=True)

In [32]:
# % correct per slice on test: original features vs. missing-number-as-"0", one table per model
for model in ret:
    original = review_typical_slices(ot_test, ret[model]["test"][0], ot_test_label)
    as_zero = review_typical_slices(ot_test, ret_zero[model]["test"][0], ot_test_label)
    table = original[["slice", "n"]].assign(original=original["% correct"],
                                            missing_as_zero=as_zero["% correct"])
    print(f"{'='*20} {model} {'='*20}")
    print(table.to_string(index=False))
    print()

==================== LogisticRegression ====================
                                   slice     n original missing_as_zero
                    positives: identical  3543    99.9%           99.9%
                positives: non-identical   843    50.5%           44.8%
   neg: case1 (same street, diff number) 14448    99.1%          100.0%
neg: case2 (same number, similar street)    15    66.7%           66.7%
                       neg: case3 (easy) 23942   100.0%          100.0%
 neg: case2s (swapped look-alike street)  1053    74.4%           72.7%
                      overall (accuracy) 43844    98.1%           98.3%

==================== RandomForestClassifier ====================
                                   slice     n original missing_as_zero
                    positives: identical  3543   100.0%          100.0%
                positives: non-identical   843    70.1%           67.7%
   neg: case1 (same street, diff number) 14448    99.6%           99.8%
neg: case

In [33]:
# % correct on test by house-number status: one row per model x feature set, one column per group
has_a, has_b = ot_test["number_a"].notna(), ot_test["number_b"].notna()
same = ot_test["number_a"] == ot_test["number_b"]   # NaN == NaN is False, so only both-present pairs can be True
groups = {
    "both present, same": has_a & has_b & same,
    "one side missing": has_a != has_b,
    "both present, different": has_a & has_b & ~same,
    "both missing": ~has_a & ~has_b,
}
# Follow-up: "one side missing" split by neg_type (case0 = positives)
for neg_type in ["case0", "case1", "case2", "case2s", "case3"]:
    mask = (has_a != has_b) & (ot_test["neg_type"] == neg_type)
    if mask.any():
        groups[f"one side missing, {neg_type}"] = mask
labels = pd.Series(ot_test_label, index=ot_test.index)

rows = {}
for model in ret:
    for setting, results in [("original", ret), ("missing as 0", ret_zero)]:
        correct = pd.Series(results[model]["test"][0], index=ot_test.index) == labels
        rows[f"{model} ({setting})"] = {g: f"{correct[mask].mean() * 100:.1f}%" for g, mask in groups.items()}
number_table = pd.DataFrame(rows).T
number_table.loc["n"] = {g: int(mask.sum()) for g, mask in groups.items()}
number_table

,"both present, same",one side missing,"both present, different",both missing,"one side missing, case0","one side missing, case1","one side missing, case2","one side missing, case3"
LogisticRegression (original),89.7%,84.7%,99.9%,96.0%,33.5%,66.0%,100.0%,99.7%
LogisticRegression (missing as 0),89.5%,89.1%,99.9%,92.0%,0.0%,100.0%,100.0%,100.0%
RandomForestClassifier (original),95.5%,87.2%,99.9%,92.0%,27.4%,82.0%,100.0%,98.9%
RandomForestClassifier (missing as 0),95.6%,89.4%,99.9%,96.0%,11.6%,95.9%,100.0%,100.0%
XGBClassifier (original),95.6%,87.6%,99.9%,92.0%,22.6%,84.9%,100.0%,99.3%
XGBClassifier (missing as 0),95.3%,89.5%,99.9%,96.0%,11.6%,96.5%,100.0%,100.0%
n,5291,1499,37029,25,164,344,1,990


In [34]:
# case1 pairs (val) where exactly one side has no house number: random sample of 20
one_missing = ot_val["number_a"].notna() != ot_val["number_b"].notna()
case1_missing = ot_val[one_missing & (ot_val["neg_type"] == "case1")]
case1_missing[["address_a", "address_b", "number_a", "number_b", "street_a", "street_b"]].sample(20, random_state=0)


,address_a,address_b,number_a,number_b,street_a,street_b
14542,74 BEACH RD,BEACH RD,74,NaN,BEACH RD,BEACH RD
14362,1101 BEDFORD ST,THIRD STREET,1101,NaN,BEDFORD ST,THIRD STREET
6443,388 HOPMEADOW ST,HOPMEADOW ST,388,NaN,HOPMEADOW ST,HOPMEADOW ST
15580,160 SOUTH ST,LITCHFIELD CENTER,160,NaN,SOUTH ST,LITCHFIELD CENTER
5204,932 DANBURY RD,SCHOOL RD,932,NaN,DANBURY RD,SCHOOL RD
15862,345 FAN HILL RD,FAN HILL RF,345,NaN,FAN HILL RD,FAN HILL RF
13009,117 LONG HILL CROSS RD,STE 2,117,NaN,LONG HILL CROSS RD,STE 2
13692,100 CT-37,ROUTE 37,100,NaN,CT-37,ROUTE 37
7009,113 POST RD,POST RD,113,NaN,POST RD,POST RD
7646,839 RIVER RD,RIVER ROAD,839,NaN,RIVER RD,RIVER ROAD


In [35]:
# Same table (test), case1 rows only (same street, different number): % rejected per number group
is_case1 = ot_test["neg_type"] == "case1"

rows = {}
for model in ret:
    for setting, results in [("original", ret), ("missing as 0", ret_zero)]:
        correct = pd.Series(results[model]["test"][0], index=ot_test.index) == labels
        rows[f"{model} ({setting})"] = {g: f"{correct[mask & is_case1].mean() * 100:.1f}%" for g, mask in groups.items()
                                        if (mask & is_case1).any()}

case1_table = pd.DataFrame(rows).T
case1_table.loc["n"] = {g: int((mask & is_case1).sum()) for g, mask in groups.items() if (mask & is_case1).any()}
case1_table

,one side missing,"both present, different","one side missing, case1"
LogisticRegression (original),66.0%,100.0%,66.0%
LogisticRegression (missing as 0),100.0%,100.0%,100.0%
RandomForestClassifier (original),82.0%,100.0%,82.0%
RandomForestClassifier (missing as 0),95.9%,99.9%,95.9%
XGBClassifier (original),84.9%,100.0%,84.9%
XGBClassifier (missing as 0),96.5%,100.0%,96.5%
n,344,14104,344


In [36]:
# Feature importances, original vs. missing-as-"0" features (one column per model x feature set).
# RF: mean impurity decrease; XGB: its default (gain); both sum to 1 per model.
# LogReg: coefficients (sign = direction, size depends on the feature's scale), so in its own table.
tree_importance = pd.DataFrame({
    f"{model} ({setting})": pd.Series(fitted[model].feature_importances_, index=fitted[model].feature_names_in_)
    for model in ["RandomForestClassifier", "XGBClassifier"]
    for setting, fitted in [("original", models), ("missing as 0", models_zero)]
}).sort_values("RandomForestClassifier (original)", ascending=False)

logreg_coef = pd.DataFrame({
    f"LogisticRegression ({setting})": pd.Series(fitted["LogisticRegression"].coef_[0], index=fitted["LogisticRegression"].feature_names_in_)
    for setting, fitted in [("original", models), ("missing as 0", models_zero)]
})

display(tree_importance.round(3))
display(logreg_coef.round(2))

,RandomForestClassifier (original),RandomForestClassifier (missing as 0),XGBClassifier (original),XGBClassifier (missing as 0)
lv_address_sim,0.196,0.207,0.001,0.002
jw_address_sim,0.127,0.121,0.809,0.797
jaccard_address,0.109,0.093,0.029,0.008
cosine_address,0.089,0.122,0.000,0.001
tversky_address,0.081,0.071,0.000,0.000
overlap_street,0.065,0.061,0.062,0.075
sorenson_address,0.063,0.094,0.000,0.000
overlap_address,0.057,0.017,0.012,0.036
jw_street_sim,0.051,0.062,0.005,0.007
lv_street_sim,0.035,0.043,0.003,0.004


,LogisticRegression (original),LogisticRegression (missing as 0)
cosine_address,-6.05,-14.42
cosine_street,-9.82,-9.20
cosine_zip,1.71,1.79
hm_zipcode_sim,0.51,1.06
jaccard_address,7.62,12.56
jaccard_street,12.44,11.47
jaccard_zip,-0.61,-1.66
jw_address_sim,2.00,-0.43
jw_street_sim,-4.14,-7.43
jw_zipcode_sim,-2.23,0.22


In [37]:
sample = find_random_sample_given_mask(ot_val, ret["LogisticRegression"]["val"][0], ot_val_label, 0)
print(sample)

==========positives: identical==========
                                      id_a  \
1600  5868b3c4-0602-4620-9e88-0f18797b9450   
812   84197d30-5ef7-4802-a2ca-e46cbd2a59f5   
795   4c9f6182-fc0d-486e-878f-80ce0cbc17f9   
3372  bdbf5611-3a3b-4457-a77f-d92c5352bf65   
2073  87e10df9-26e3-4981-8257-57d0b92e1cef   
1883  1bd5778c-2503-4066-b8ff-375ba4eda710   
3233  9333ade1-7436-488a-810e-db978dc8582e   
928   030a63df-24ec-4e3a-bcd6-a79169f7fd5d   
2383  e584e858-1810-43a5-b89f-6f06a5847601   
434   07aa789b-e139-4e5a-8f27-03aadcb39b6e   

                                      id_b           address_a  \
1600  5868b3c4-0602-4620-9e88-0f18797b9450    221 BULL HILL LN   
812   84197d30-5ef7-4802-a2ca-e46cbd2a59f5   565 LONG HILL AVE   
795   4c9f6182-fc0d-486e-878f-80ce0cbc17f9  484 BRIDGEPORT AVE   
3372  bdbf5611-3a3b-4457-a77f-d92c5352bf65         80 DOYLE RD   
2073  87e10df9-26e3-4981-8257-57d0b92e1cef      1799 SUMMER ST   
1883  1bd5778c-2503-4066-b8ff-375ba4eda710   883 HIGH RI

### Results — Overture → Overture, val (refreshed 2026-10-16)

23 features (22 similarities + `number_missing`), default hyperparameters, no class weighting. Numbers are **% of the slice handled correctly** on val (found for matches, rejected for non-matches), as printed by the `review_typical_slices` cell above. Test numbers and slice sizes: `results.md` (repo root).

| model | identical matches found | **non-identical matches found** | case1 rejected | case2 rejected | easy rejected |
|---|---|---|---|---|---|
| LogReg | 100.0 | **81.8** | 99.8 | 2.9 | 100.0 |
| RF | 100.0 | **85.4** | 99.5 | 23.5 | 100.0 |
| XGBoost | 100.0 | **84.9** | 99.5 | 20.6 | 100.0 |

- The old headline F1 0.97 was inflated by identical-string positives (~79% of positives, ~all found); the non-identical column is the one that matters.
- **Case 2** is only 34 val rows and contains probable label noise (open question, hand-check pending) — don't interpret it yet.
- **Visible error sources:** text before the number on side B → `number_missing`; `ST`/`STREET` suffix variants.

*Diagnostic queries + results for this section: **Diagnostics** at the bottom of this notebook → Overture baseline.*

## Diagnostics

One-off checks kept out of the pipeline above. Paths are relative to this notebook's folder.

### House-number regex

**Problem:** The old `^(\d+)\s+` pattern missed numbers like `12A` and `16-18`, leaving them in the street name.  
**Check:** New regex on hand-picked cases; streets still starting with a digit; raw addresses the old pattern missed.  
**Result:** Hand cases correct; 473 / 492 streets start with a digit (ordinals, fractions, double numbers); old pattern missed 1,202 / 1,484 letter suffixes and 444 / 742 ranges (A / B) (re-run 2026-10-13)

```python
# Test the house-number token regex on hand-picked cases
check = pd.DataFrame({"address": ["12 MAIN ST", "12A MAIN ST", "16-18 ELM ST", "20-A ELM ST"]})
check["check"] = check["address"].str.extract(r"^(\d[A-Z0-9a-z-]*)\s")
print(check)

# After parsing: sample + count of streets that still start with a digit (ordinal streets, 1/2 addresses, double numbers)
ret = pd.concat([ot_train, ot_val, ot_test])
print(ret[ret["street_a"].str.match(r"^\d")][["street_a"]].sample(frac=1).head(10))
print(ret["street_a"].str.match(r"^\d").sum())
print(ret[ret["street_b"].str.match(r"^\d")][["street_b"]].sample(frac=1).head(10))
print(ret["street_b"].str.match(r"^\d").sum())

# Raw addresses the old ^(\d+)\s+ pattern missed: letter suffix (12A) on A / B, then ranges (16-18) on A / B
print(ret["address_a"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_b"].str.match(r"^\d+[A-Za-z]+\s").sum())
print(ret["address_a"].str.match(r"^[0-9]+-[0-9]+\s").sum())
print(ret["address_b"].str.match(r"^[0-9]+-[0-9]+\s").sum())
```

### Streets under 2 characters

**Problem:** The Overlap bigram metric divides by zero on strings shorter than 2 characters.  
**Check:** Find any short street names.  
**Result:** Test split only: street `1` (7 on A, 11 on B) → guarded in `features.py` (re-run 2026-10-13)

```python
# Streets under 2 characters (break bigram metrics): only the test addresses that are literally "1"
for df in [ot_train, ot_val, ot_test]:
    print(df[df["street_a"].str.len() < 2]["street_a"])
    print(df[df["street_b"].str.len() < 2]["street_b"])
    print()
```

### Positives with identical addresses

**Problem:** Suspected the models were misclassifying positives whose addresses are identical.  
**Check:** Misclassifications within `case0`, split by identical vs. different address and by number status.  
**Result:** Identical: LR 1 / 3,390 wrong (a ZIP disagreement), RF 0, XGB 0. Different: 82–85% correct. The `number_missing` subset is empty by construction since `both_number_missing` was split out; use `both_number_missing` instead (re-run 2026-10-13)

```python
print([k for k in ret.keys()])
mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] == ot_val["address_b"])).values
totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
print([int(c) for c in totals])

# missing = [ot_val[mask][(preds != ot_val_label) & (ot_val["number_missing"] == False)] for model, (preds, probs) in ret.items()]
# for miss,model in zip(missing,[k for k in ret.keys()]):
#     print(f"{'='*20} {model} {'='*20}")
#     print(miss[["address_a", "address_b", "ZipCode_a", "ZipCode_b"]])


# mask = ((ot_val["neg_type"] == "case0") & (ot_val["address_a"] != ot_val["address_b"])).values
# totals = [(preds[mask] == ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([int(c) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == True)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# mask = ((ot_val["neg_type"] == "case0") & 
#         (ot_val["address_a"] == ot_val["address_b"]) & 
#         (ot_val['number_missing'] == False)).values
# n = mask.sum()
# totals = [(preds[mask] != ot_val_label[mask]).sum() for model, (preds, probs) in ret.items()]
# print([float(c/n) for c in totals])

# print(
# len(ot_val[
#     (ot_val["neg_type"] == "case0") & 
#     (ot_val["address_a"] == ot_val["address_b"]) & 
#     (ot_val['number_missing'] == True)
# ]))
```